In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic data

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40255413,0,Is gastrointestinal (GI) problems more common ...
1,40255413,1,Does autism spectrum disorders affect a child'...
2,40255413,2,What is one possible underlying factor contrib...
3,40255413,3,What is a common challenge that many individua...
4,40255413,4,How do gastrointestinal (GI) symptoms affect t...
...,...,...,...
495,40938164,0,Is autism a natural part of human diversity?
496,40938164,1,Can individuals with autism spectrum disorders...
497,40938164,2,What is it about individuals with autism spect...
498,40938164,3,What is the main difference between autism and...


# Demo retriever

In [4]:
# select question to demonstrate
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)

Is gastrointestinal (GI) problems more common in individuals with autism spectrum disorders than in the general population?


In [5]:
# demonstrate vector search
rag.vsearch(demo_query, num_results=2)

[{'pmid': '40255413',
  'elocationid': 'pii: 02537176251331152',
  'title': 'Gastrointestinal Manifestations and Associated Comorbidities in Children with Autism Spectrum Disorder: A Cross-sectional Analysis from South India.',
  'journal': 'Indian journal of psychological medicine',
  'year': '2025',
  'author': 'Nishant Prabhakaran, Sruthy Jestine, Suhas Chandran, Lakshmi Shiva, Ann Maria Moncy',
  'affiliation': "Centre for Advanced Research and Excellence in Autism and Developmental Disorders, St. John's Medical College Hospital and Research Institute, Bengaluru, Karnataka, India. Child and Adolescent Psychiatry Unit, St. John's Medical College Hospital, Bengaluru, Karnataka, India.",
  'abstract': 'Gastrointestinal (GI) symptoms are frequently reported in children with autism spectrum disorder (ASD) and may significantly impact behavior, sleep, adaptive functioning, and the severity of autism. This study aims to explore the relationship between GI symptoms and these factors in chi

In [6]:
# demonstrate keyword search
rag.kwsearch(demo_query, num_results=2)

[{'pmid': '40313537',
  'elocationid': 'pii: 1552369',
  'title': '',
  'journal': 'Frontiers in neuroscience',
  'year': '2025',
  'author': 'Gari L Eberly, Marie Manthey, Karen K L Pang, Heba Hussein, Emmanuel Vargas Paniagua, Scott Machen, Sara Maeve Klingensmith, Polina Anikeeva',
  'affiliation': 'Department of Electrical Engineering and Computer Science, Massachusetts Institute of Technology, Cambridge, MA, United States. MIT-Harvard Graduate Program in Health Sciences and Technology, Boston, MA, United States. K. Lisa Yang Brain-Body Center, Massachusetts Institute of Technology, Cambridge, MA, United States. Department of Biology, Wellesley College, Wellesley, MA, United States.',
  'abstract': 'Gastrointestinal (GI) comorbidities are common among those with Autism Spectrum Disorder (ASD), but their etiology is not well understood. This study aimed to characterize gastrointestinal morphology and function in Shank3B mutant mice, a common genetic model of ASD, to identify potenti

# Evaluate retriever, by hit rate and mean reciprocal rank (MRR)

## Define functions

In [7]:
def get_relevance_total(synth_records, search_function, num_results):
    # initialize relevance total
    relevance_total = []
    # for each document, make relevance
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        true_pmid = record['pmid']
        search_results = search_function(query, num_results)
        relevance = [true_pmid==d['pmid'] for d in search_results]
        relevance_total.append(relevance)
    return relevance_total

In [8]:
def hit_rate(relevance_total):
    hit_boolean = [True in line for line in relevance_total]
    numerator = sum(hit_boolean)
    denominator = len(hit_boolean)
    return numerator/denominator

In [9]:
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1/(rank+1)
    return total_score/len(relevance_total)

In [10]:
def get_questions_by_correctness(synth_records, relevance_total):
    questions_answered_correct = []
    questions_answered_incorrect = []
    for i in range(len(synth_records)):
        relevance = relevance_total[i]
        question = synth_records[i]['synthetic_question']
        if True in relevance:
            questions_answered_correct.append(question)
        else:
            questions_answered_incorrect.append(question)
    all_questions = {'correct' : questions_answered_correct, \
    'incorrect' : questions_answered_incorrect}
    return all_questions

## Work

In [11]:
# cast as list of dictionaries
synth_records = df_synth.to_dict('records')
print(len(synth_records))

500


In [12]:
# number of documents retrieved
num_results = 5
print(num_results)

5


In [13]:
# for vector search, get relevance total
print(datetime.now())
relevance_total_v = get_relevance_total(synth_records=synth_records, \
search_function=rag.vsearch, num_results=num_results)
print(datetime.now())

2025-09-14 00:54:32.021730


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-14 00:55:34.074019


In [14]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_v)

{'hit rate': 0.468, 'mrr': 0.3644999999999999}

In [15]:
# for vector search, see questions correctly and incorrectly answered
qbc_v = get_questions_by_correctness(synth_records, relevance_total_v)
print('---correct---')
print('\n'.join(qbc_v['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_v['incorrect'][:10]))

---correct---
Is gastrointestinal (GI) problems more common in individuals with autism spectrum disorders than in the general population?
What is one possible underlying factor contributing to the co-occurrence of gastrointestinal (GI) symptoms in children with autism spectrum disorder (ASD)?
How do gastrointestinal (GI) symptoms affect the daily lives and behaviors of individuals with autism spectrum disorder?
What are some ways that researchers can better understand the impact of Dialectical Behavior Therapy (DBT) on individuals with autism spectrum disorders who experience extreme emotional difficulties?
Can researchers use Real-time data from Ecological Momentary Assessments (EMAs) to inform and improve the effectiveness of DBT for autistic adults who struggle with emotion regulation?
How do some individuals on the autism spectrum experience emotional regulation issues that may impact their response to DBT therapy?
What are the potential benefits and drawbacks of using artificial i

In [16]:
# for keyword search, get relevance total
print(datetime.now())
relevance_total_kw = get_relevance_total(synth_records=synth_records, \
search_function=rag.kwsearch, num_results=num_results)
print(datetime.now())

2025-09-14 00:55:34.098647


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-14 00:55:36.452074


In [17]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_kw)

{'hit rate': 0.472, 'mrr': 0.38520000000000004}

In [18]:
# for keyword search, seee questions correctly and incorrectly answered
qbc_kw = get_questions_by_correctness(synth_records, relevance_total_kw)
print('---correct---')
print('\n'.join(qbc_kw['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_kw['incorrect'][:10]))

---correct---
Is gastrointestinal (GI) problems more common in individuals with autism spectrum disorders than in the general population?
What is one possible underlying factor contributing to the co-occurrence of gastrointestinal (GI) symptoms in children with autism spectrum disorder (ASD)?
How do gastrointestinal (GI) symptoms affect the daily lives and behaviors of individuals with autism spectrum disorder?
Is there a difference between the prevalence of autism spectrum disorder and the prevalence of other conditions that can also affect sexual experiences?
What are some ways that researchers can better understand the impact of Dialectical Behavior Therapy (DBT) on individuals with autism spectrum disorders who experience extreme emotional difficulties?
Can researchers use Real-time data from Ecological Momentary Assessments (EMAs) to inform and improve the effectiveness of DBT for autistic adults who struggle with emotion regulation?
What are the key challenges in conducting Ecolo

In [19]:
print(datetime.now())

2025-09-14 00:55:36.466141
